# NQAI Voice — Platform v0.1 Server on Colab

Boots the FastAPI server (`server.main:app`) on Colab Pro (T4 or A100) and exposes a public HTTPS URL through Cloudflare's free quick-tunnel — no account or signup needed.

**Run order:** cell 1 → kernel restart → cell 3+ (skip cell 2 markdown). Total time on T4: ~6-8 min from cold start to a working `https://*.trycloudflare.com` URL.

**Repo:** `neeko-voice/notebooks/03-platform-server-colab.ipynb`

## 1. Clone repo + install (~3-5 min)

We pin every dep that Chatterbox-TTS conflicts with on Colab; the upstream wheels declare strict equality constraints we can't satisfy with the runtime's defaults. Same recipe as `02-chatterbox-voice-clone-colab.ipynb`.

In [ ]:
import os, subprocess

REPO_URL = os.environ.get('NQAI_REPO_URL', 'https://github.com/erdalgumuss/neeko-voice.git')
REPO_BRANCH = os.environ.get('NQAI_REPO_BRANCH', 'main')
REPO_DIR = '/content/neeko-voice'

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--depth=1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR])

%cd $REPO_DIR

!pip uninstall -y torchvision
!pip install -q chatterbox-tts --no-deps
!pip install -q torch==2.6.0 torchaudio==2.6.0
!pip install -q transformers==4.46.3 tokenizers==0.20.3
!pip install -q librosa==0.11.0 resemble-perth==1.0.1 s3tokenizer conformer==0.3.2 diffusers==0.29.0 safetensors==0.5.3 omegaconf pyloudnorm einops
!pip install -q 'numpy==2.0.2' --force-reinstall --no-deps
!pip install -q fastapi 'uvicorn[standard]' python-multipart pydantic pyyaml httpx soundfile
print('=' * 60)
print('Install done — RESTART KERNEL now')
print('=' * 60)

## 2. RESTART KERNEL

Runtime → Restart runtime. Then jump to cell 3 (do **not** re-run cell 1).

## 3. Verify GPU + pinned versions

In [ ]:
import torch, numpy as np, transformers

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'torch={torch.__version__}, numpy={np.__version__}, transformers={transformers.__version__}')

## 4. Stage reference audio + seed catalog

The repo ships with the NEEKO bridge reference manifest, but the actual MP3 lives outside git. Either upload `neeko-v0.1-reference.mp3` to your Google Drive root and mount it here, or skip — the server will start with an empty catalog and you can enroll voices via `POST /v1/voices` once the tunnel is live.

In [ ]:
import os, shutil
%cd /content/neeko-voice

REF_TARGET = 'data/reference-audio/neeko-v0.1-reference.mp3'
if not os.path.isfile(REF_TARGET):
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        src = '/content/drive/MyDrive/neeko-v0.1-reference.mp3'
        if os.path.isfile(src):
            os.makedirs(os.path.dirname(REF_TARGET), exist_ok=True)
            shutil.copy(src, REF_TARGET)
            print(f'staged: {REF_TARGET}')
        else:
            print(f'OPTIONAL — {src} not found; NEEKO slot will be unusable until you enroll a reference')
    except Exception as e:
        print(f'skipped Drive mount: {e}')
else:
    print(f'already staged: {REF_TARGET}')

## 5. Configure + boot the server in the background

Auth is on by default. We mint two random API keys here — copy them somewhere safe; you'll need them for every request below.

In [ ]:
import os, secrets, subprocess, time
%cd /content/neeko-voice

ADMIN_KEY = 'nqai-admin-' + secrets.token_urlsafe(24)
DEV_KEY   = 'nqai-dev-'   + secrets.token_urlsafe(24)

env = os.environ.copy()
env['NQAI_API_KEYS']       = f'{ADMIN_KEY},{DEV_KEY}'
env['NQAI_REQUIRE_AUTH']   = 'true'
env['NQAI_DEVICE']         = 'cuda'
env['NQAI_PORT']           = '8000'
env['PYTHONPATH']          = '/content/neeko-voice/src'

log = open('/content/server.log', 'wb')
proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server.main:app', '--host', '0.0.0.0', '--port', '8000'],
    env=env, stdout=log, stderr=subprocess.STDOUT,
)
print(f'server pid: {proc.pid}')
print()
print(f'ADMIN_KEY = {ADMIN_KEY}')
print(f'DEV_KEY   = {DEV_KEY}')

# wait for /health to respond
import httpx
for _ in range(40):
    try:
        r = httpx.get('http://127.0.0.1:8000/health', timeout=2)
        if r.status_code == 200:
            print('\n/health:', r.json())
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print('\nserver did not become ready — tail of log:')
    !tail -40 /content/server.log

## 6. Warm up the model (loads ~3 GB of weights, T4 takes ~60 s)

In [ ]:
import httpx, time

t0 = time.time()
r = httpx.post('http://127.0.0.1:8000/admin/warmup',
               headers={'Authorization': f'Bearer {ADMIN_KEY}'}, timeout=600)
print(f'warmup HTTP {r.status_code} in {time.time()-t0:.1f}s')
print(r.json() if r.status_code == 200 else r.text)

## 7. Open a public Cloudflare quick-tunnel

`cloudflared --url http://localhost:8000` issues an ephemeral `https://*.trycloudflare.com` hostname that proxies straight to the local server. No account, no auth, no per-request limit beyond Cloudflare's normal abuse protections. The URL dies with the Colab session — for a stable URL use a named tunnel or paid ngrok.

In [ ]:
import os, subprocess, time, re, pathlib

if not os.path.isfile('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

tunnel_log = '/content/cloudflared.log'
pathlib.Path(tunnel_log).write_text('')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000'],
    stdout=open(tunnel_log, 'wb'), stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(40):
    log_text = pathlib.Path(tunnel_log).read_text(errors='ignore')
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
    if m:
        public_url = m.group(0)
        break
    time.sleep(1)

print(f'PUBLIC URL: {public_url}')
print(f'DEV_KEY   : {DEV_KEY}')
print()
print('# Try it:')
print(f"curl -H 'Authorization: Bearer {DEV_KEY}' {public_url}/v1/voices")

## 8. Quick test from inside the notebook

We POST a Türkçe sentence and play the resulting WAV inline. If the catalog is empty, this cell will first enroll the NEEKO bridge voice from the staged MP3.

In [ ]:
import httpx, os, json
from IPython.display import Audio, display

BASE = public_url
H = {'Authorization': f'Bearer {DEV_KEY}'}

catalog = httpx.get(f'{BASE}/v1/voices', headers=H, timeout=30).json()
print('catalog:', json.dumps(catalog, ensure_ascii=False, indent=2))

if catalog['count'] == 0 and os.path.isfile('data/reference-audio/neeko-v0.1-reference.mp3'):
    with open('data/reference-audio/neeko-v0.1-reference.mp3', 'rb') as f:
        r = httpx.post(f'{BASE}/v1/voices', headers=H,
            data={'voice_id': 'neeko-v01', 'display_name': 'NEEKO v0.1 (köprü)',
                  'gender': 'neutral', 'style_tags': 'warm,child-directed'},
            files={'reference_audio': ('neeko.mp3', f, 'audio/mpeg')}, timeout=120)
    print('enrolled:', r.json())
    catalog = httpx.get(f'{BASE}/v1/voices', headers=H).json()

if catalog['count']:
    vid = catalog['voices'][0]['voice_id']
    r = httpx.post(f'{BASE}/v1/tts', headers=H, timeout=120,
        json={'text': 'Merhaba, ben Neeko. Bugün seninle ne oynayalım?', 'voice_id': vid})
    out = f'/content/test_{vid}.wav'
    open(out, 'wb').write(r.content)
    print('headers:', {k: v for k, v in r.headers.items() if k.startswith('X-NQAI-')})
    display(Audio(out))
else:
    print('catalog still empty — enroll a voice via POST /v1/voices first')

## 9. Run the 10-sentence smoke set against every voice

Writes per-voice WAVs to `/content/experiments/<date>-platform-smoke/output/<voice_id>/`.

In [ ]:
!PYTHONPATH=src python scripts/smoke_test.py \
    --base-url $public_url \
    --api-key $DEV_KEY \
    --out /content/experiments/platform-smoke

## 10. Tail server log if anything misbehaves

In [ ]:
!tail -50 /content/server.log